In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [2]:
model_name = "typeform/distilbert-base-uncased-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = {int(k): v for k, v in model.config.id2label.items()}
entailment_id = next(i for i, label in id2label.items() if "entail" in label.lower())
contradiction_id = next(i for i, label in id2label.items() if "contrad" in label.lower())

print("model:", model_name)
print("id2label:", id2label)
print("entailment_id:", entailment_id, "label:", id2label[entailment_id])
print("contradiction_id:", contradiction_id, "label:", id2label[contradiction_id])


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

model: typeform/distilbert-base-uncased-mnli
id2label: {0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}
entailment_id: 0 label: ENTAILMENT
contradiction_id: 2 label: CONTRADICTION


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
batch_size = 64
margin_12_all = []
margin_21_all = []
min_margin_all = []

with torch.no_grad():
    for i in tqdm(range(0, len(ds), batch_size)):
        batch_s1 = sent1[i:i + batch_size]
        batch_s2 = sent2[i:i + batch_size]

        enc_12 = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        enc_21 = tokenizer(
            batch_s2,
            batch_s1,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )

        enc_12 = {k: v.to(device) for k, v in enc_12.items()}
        enc_21 = {k: v.to(device) for k, v in enc_21.items()}

        logits_12 = model(**enc_12).logits
        logits_21 = model(**enc_21).logits

        margin_12 = (logits_12[:, entailment_id] - logits_12[:, contradiction_id]).detach().cpu().numpy()
        margin_21 = (logits_21[:, entailment_id] - logits_21[:, contradiction_id]).detach().cpu().numpy()
        min_margin = np.minimum(margin_12, margin_21)

        margin_12_all.extend(margin_12.tolist())
        margin_21_all.extend(margin_21.tolist())
        min_margin_all.extend(min_margin.tolist())

margin_12_all = np.array(margin_12_all)
margin_21_all = np.array(margin_21_all)
min_margin_all = np.array(min_margin_all)
y_pred = (min_margin_all > 0).astype(int)

print("done")
print("predicted_positive_rate:", float(y_pred.mean()))


  0%|          | 0/7 [00:00<?, ?it/s]

done
predicted_positive_rate: 0.3235294117647059


In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.5857843137254902, 'f1': 0.5888077858880778}
                precision    recall  f1-score   support

not_paraphrase       0.43      0.91      0.58       129
    paraphrase       0.92      0.43      0.59       279

      accuracy                           0.59       408
     macro avg       0.67      0.67      0.59       408
  weighted avg       0.76      0.59      0.59       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("margin_12:", float(margin_12_all[i]))
    print("margin_21:", float(margin_21_all[i]))
    print("min_margin:", float(min_margin_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
margin_12: 7.967843055725098
margin_21: 6.533724784851074
min_margin: 6.533724784851074
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
margin_12: -9.776514053344727
margin_21: -0.8201401233673096
min_margin: -9.776514053344727
true: 0 pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
margin_12: -0.6154601573944092
margin_21: 6.547325134277344
min_

In [7]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("margin_12:", float(margin_12_all[i]))
    print("margin_21:", float(margin_21_all[i]))
    print("min_margin:", float(min_margin_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


num_errors: 169
idx: 3
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candidate before the primaries .
margin_12: -1.0258152484893799
margin_21: 6.178720474243164
min_margin: -1.0258152484893799
true: 1 pred: 0
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
margin_12: -1.392435073852539
margin_21: -1.2508978843688965
min_margin: -1.392435073852539
true: 1 pred: 0
idx: 7
sentence1: This integrates with Rational PurifyPlus and allows developers to work in supported versions of Java , Visual C # and Visual Basic .NET.
sentence2: IBM said the Rational products were also integrated with Rational PurifyPlus , which allows developers to work in Java ,

In [8]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(device),
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
    "method": "mnli_bidirectional_min_margin",
}
summary


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.5857843137254902,
 'f1': 0.5888077858880778,
 'method': 'mnli_bidirectional_min_margin'}